In [2]:
import numpy as np
import pandas as pd
import time
import pickle
import warnings
from itertools import product
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import (
    accuracy_score, precision_score,
    recall_score, f1_score,
    roc_auc_score, roc_curve
)
from xgboost import XGBClassifier

warnings.filterwarnings("ignore")


# ============================================================
# LOAD DATA
# ============================================================

def load_cleaned_data(filename='data_cleaned.pkl'):
    print("Loading cleaned data...")
    with open(filename, 'rb') as f:
        data = pickle.load(f)
    return data


# ============================================================
# FIND OPTIMAL THRESHOLD
# ============================================================

def find_optimal_threshold(y_true, y_proba):
    fpr, tpr, thresholds = roc_curve(y_true, y_proba)
    J = tpr - fpr
    return thresholds[np.argmax(J)]


# ============================================================
# CROSS VALIDATION
# ============================================================

def run_cross_validation(X, y, preprocessor, xgb_params, n_splits=10):

    skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=42)

    results = []

    for fold, (train_idx, test_idx) in enumerate(skf.split(X, y), 1):

        start_time = time.time()

        X_train_raw = X.iloc[train_idx]
        X_test_raw = X.iloc[test_idx]
        y_train = y[train_idx]
        y_test = y[test_idx]

        X_train = preprocessor.fit_transform(X_train_raw)
        X_test = preprocessor.transform(X_test_raw)

        model = XGBClassifier(
            n_estimators=xgb_params['n_estimators'],
            max_depth=xgb_params['max_depth'],
            learning_rate=xgb_params['learning_rate'],
            subsample=xgb_params['subsample'],
            colsample_bytree=xgb_params['colsample_bytree'],
            reg_lambda=xgb_params['reg_lambda'],
            random_state=xgb_params['random_seed'],
            use_label_encoder=False,
            eval_metric='logloss'
        )

        model.fit(X_train, y_train)

        y_proba = model.predict_proba(X_test)[:, 1]

        threshold = find_optimal_threshold(y_test, y_proba)
        y_pred = (y_proba >= threshold).astype(int)

        results.append({
            "Fold": fold,
            "Accuracy": accuracy_score(y_test, y_pred),
            "Precision": precision_score(y_test, y_pred, zero_division=0),
            "Recall": recall_score(y_test, y_pred, zero_division=0),
            "F1 Score": f1_score(y_test, y_pred, zero_division=0),
            "AUC": roc_auc_score(y_test, y_proba),
            "Threshold": threshold,
            "Training Time (s)": time.time() - start_time
        })

    return pd.DataFrame(results)


# ============================================================
# GRID SEARCH
# ============================================================

def grid_search_xgb(X, y, preprocessor, param_grid, n_splits=10):

    combinations = list(product(
        param_grid['n_estimators'],
        param_grid['max_depth'],
        param_grid['learning_rate'],
        param_grid['subsample'],
        param_grid['colsample_bytree'],
        param_grid['reg_lambda'],
        param_grid['random_seed']
    ))

    all_results = []

    print("\nGRID SEARCH STARTED")
    print("=" * 60)

    for i, params in enumerate(combinations, 1):

        print(f"[{i}/{len(combinations)}] Testing {params}")

        xgb_params = {
            "n_estimators": params[0],
            "max_depth": params[1],
            "learning_rate": params[2],
            "subsample": params[3],
            "colsample_bytree": params[4],
            "reg_lambda": params[5],
            "random_seed": params[6]
        }

        cv_df = run_cross_validation(X, y, preprocessor, xgb_params, n_splits)

        all_results.append({
            "n_estimators": xgb_params["n_estimators"],
            "max_depth": xgb_params["max_depth"],
            "learning_rate": xgb_params["learning_rate"],
            "subsample": xgb_params["subsample"],
            "colsample_bytree": xgb_params["colsample_bytree"],
            "reg_lambda": xgb_params["reg_lambda"],
            "Accuracy": cv_df["Accuracy"].mean(),
            "Precision": cv_df["Precision"].mean(),
            "Recall": cv_df["Recall"].mean(),
            "F1 Score": cv_df["F1 Score"].mean(),
            "AUC": cv_df["AUC"].mean()
        })

    results_df = pd.DataFrame(all_results)
    results_df = results_df.sort_values(
        by=["F1 Score", "Recall"],
        ascending=False
    ).reset_index(drop=True)

    return results_df


# ============================================================
# STABILITY
# ============================================================

def calculate_stability(cv_df):

    metrics = ["Accuracy", "Precision", "Recall", "F1 Score", "AUC"]

    summary = []

    for metric in metrics:
        mean_val = cv_df[metric].mean()
        std_val = cv_df[metric].std()
        cv_percent = (std_val / mean_val) * 100 if mean_val != 0 else 0

        if cv_percent < 5:
            status = "SANGAT STABIL"
        elif cv_percent < 10:
            status = "STABIL"
        else:
            status = "KURANG STABIL"

        summary.append({
            "Metric": metric,
            "Mean": mean_val,
            "Std Dev": std_val,
            "CV (%)": cv_percent,
            "Stability": status
        })

    summary_df = pd.DataFrame(summary)

    time_mean = cv_df["Training Time (s)"].mean()
    time_total = cv_df["Training Time (s)"].sum()

    return summary_df, time_mean, time_total


# ============================================================
# MAIN
# ============================================================

def main():

    print("="*70)
    print("XGBOOST - HYPERPARAMETER TUNING + STABILITY")
    print("="*70)

    data_loaded = load_cleaned_data("data_cleaned.pkl")
    data_cleaned = data_loaded['data_cleaned']
    preprocessor = data_loaded['preprocessor']

    X = data_cleaned.drop(columns=["diagnosis_lanjutan"])
    y = data_cleaned["diagnosis_lanjutan"].values

    param_grid = {
        "n_estimators": [100, 200],
        "max_depth": [3, 5],
        "learning_rate": [0.01, 0.1],
        "subsample": [0.8, 1.0],
        "colsample_bytree": [0.8, 1.0],
        "reg_lambda": [1, 5],
        "random_seed": [42]
    }

    tuning_results = grid_search_xgb(X, y, preprocessor, param_grid)
    tuning_results.to_csv("xgb_grid_search_results.csv", index=False)

    best_config = tuning_results.iloc[0]

    xgb_params_best = {
        "n_estimators": int(best_config["n_estimators"]),
        "max_depth": int(best_config["max_depth"]),
        "learning_rate": float(best_config["learning_rate"]),
        "subsample": float(best_config["subsample"]),
        "colsample_bytree": float(best_config["colsample_bytree"]),
        "reg_lambda": float(best_config["reg_lambda"]),
        "random_seed": 42
    }

    print("\nBEST CONFIGURATION:")
    print(xgb_params_best)

    cv_best = run_cross_validation(X, y, preprocessor, xgb_params_best)

    print("\nDETAIL 10-FOLD CROSS VALIDATION")
    print(cv_best.to_string(index=False))

    summary_df, time_mean, time_total = calculate_stability(cv_best)

    print("\nRINGKASAN PERFORMA + STABILITAS")
    print(summary_df.to_string(index=False))

    print("\nWAKTU KOMPUTASI:")
    print(f"Rata-rata per fold : {time_mean:.4f} detik")
    print(f"Total waktu        : {time_total:.4f} detik")

    summary_df.to_csv("xgb_stability_summary.csv", index=False)

    print("\n✅ SELESAI")
    print("File disimpan:")
    print("- xgb_grid_search_results.csv")
    print("- xgb_stability_summary.csv")


if __name__ == "__main__":
    main()

XGBOOST - HYPERPARAMETER TUNING + STABILITY
Loading cleaned data...

GRID SEARCH STARTED
[1/64] Testing (100, 3, 0.01, 0.8, 0.8, 1, 42)
[2/64] Testing (100, 3, 0.01, 0.8, 0.8, 5, 42)
[3/64] Testing (100, 3, 0.01, 0.8, 1.0, 1, 42)
[4/64] Testing (100, 3, 0.01, 0.8, 1.0, 5, 42)
[5/64] Testing (100, 3, 0.01, 1.0, 0.8, 1, 42)
[6/64] Testing (100, 3, 0.01, 1.0, 0.8, 5, 42)
[7/64] Testing (100, 3, 0.01, 1.0, 1.0, 1, 42)
[8/64] Testing (100, 3, 0.01, 1.0, 1.0, 5, 42)
[9/64] Testing (100, 3, 0.1, 0.8, 0.8, 1, 42)
[10/64] Testing (100, 3, 0.1, 0.8, 0.8, 5, 42)
[11/64] Testing (100, 3, 0.1, 0.8, 1.0, 1, 42)
[12/64] Testing (100, 3, 0.1, 0.8, 1.0, 5, 42)
[13/64] Testing (100, 3, 0.1, 1.0, 0.8, 1, 42)
[14/64] Testing (100, 3, 0.1, 1.0, 0.8, 5, 42)
[15/64] Testing (100, 3, 0.1, 1.0, 1.0, 1, 42)
[16/64] Testing (100, 3, 0.1, 1.0, 1.0, 5, 42)
[17/64] Testing (100, 5, 0.01, 0.8, 0.8, 1, 42)
[18/64] Testing (100, 5, 0.01, 0.8, 0.8, 5, 42)
[19/64] Testing (100, 5, 0.01, 0.8, 1.0, 1, 42)
[20/64] Testing 